In [1]:
import pandas as pd
import numpy as np
#from sklearn import train_test_split

# select significantly reversed miRNA pairs
f = open('kazuki_data.json')
kazuki_data = pd.read_json(f, orient='split')
f.close()
f = open('watara_data.json')
watara_data = pd.read_json(f, orient='split')
f.close()
f = open('yokoi_data.json')
yokoi_data = pd.read_json(f, orient='split')
f.close()

#we need to have training/validation. 
#the article has a training split of the kazuki + yokoi (only 2000 noncancer and 200 OV)
# so im skipping test, so validation the rest of yokoi (759 noncancer,120 OV) and watara (100 noncancer, 40 OV)
#for training, I'm only going to use 'stable mirna pairs' as described in article, and need to maintain REO of like 95% or sum

training = pd.concat([kazuki_data, yokoi_data.loc[0,:].iloc[:2000], yokoi_data.loc[1,:].iloc[:200]])
validation = pd.concat([watara_data, yokoi_data.loc[0,:].iloc[2000:], yokoi_data.loc[1,:].iloc[200:]])

t_healthy = training.loc[0]
t_cancer = training.loc[1]
v_healthy = validation.loc[0]
v_cancer = validation.loc[1]

m_t_healthy = t_healthy.shape[0]
m_t_cancer = t_cancer.shape[0]
m_v_healthy = v_healthy.shape[0]
m_v_cancer = v_cancer.shape[0]

n_mirnas = t_healthy.shape[1] #gets number of columns
miRNA_names_list = list(t_healthy.columns)

print("healthy training data")
print(t_healthy.shape)
print(t_healthy.head)
print("cancer training data")
print(t_cancer.shape)
print(t_cancer.head)
print("healthy validation data")
print(v_healthy.shape)
print(v_healthy.head)
print("cancer validation data")
print(v_cancer.shape)
print(v_cancer.head)


healthy training data
(6965, 2565)
<bound method NDFrame.head of     MIMAT0000062  MIMAT0000063  MIMAT0000064  MIMAT0000065  MIMAT0000066  \
0          3.700         3.669        -2.681         3.499        -2.681   
0         -4.018        -4.018        -4.018        -4.018        -4.018   
0          1.514         0.869         0.464        -0.987        -4.325   
0         -4.417        -4.417        -4.417        -1.036        -1.897   
0         -5.541         0.598        -5.281        -3.185         0.986   
..           ...           ...           ...           ...           ...   
0         -3.515         1.626         6.450        -3.515         1.136   
0         -3.160        -3.160        -3.160        -3.160        -3.160   
0         -3.717         0.509        -3.717        -3.717         1.166   
0         -2.374        -2.374         1.426         2.128         3.553   
0         -5.741         1.007        -5.741        -5.741        -5.741   

    MIMAT0000067  MIMA

In [2]:
import numpy as np
a = np.array([1,2,3,4])
b = np.array([2,3,4,5])
np.greater(a,b)

array([False, False, False, False])

In [2]:
W = t_healthy.values
print(W.shape)

(6965, 2565)


In [5]:
CONST_k = 0.95
significant_pairs = np.zeros((n_mirnas, n_mirnas))
for i in range(0,n_mirnas):
    W_current_miRNA = W[:, i:i+1] #shape (m_samples, 1)
    W_rest_miRNA = W[:, i+1:] #shape (m_samples, n_mirnas-i-1)
    W_comparisons = W_rest_miRNA < W_current_miRNA
    W_greater = np.sum(W_comparisons, axis=0) / m_t_healthy
    W_mask_greater_correlation = np.add((W_greater >= CONST_k).nonzero(), i) #-1
    W_mask_less_correlation = np.add((W_greater <= (1-CONST_k)).nonzero(), i) #-1
    significant_pairs[i][[W_mask_greater_correlation]] = 1
    significant_pairs[i][[W_mask_less_correlation]] = -1


In [69]:
testindicies1 = significant_pairs.nonzero()
print(len(testindicies1[0]))
no_lessthantest = 0
no_greaterthantest = 0
for i in range(len(testindicies1[0])):
    if(significant_pairs[testindicies1[0][i]][testindicies1[1][i]] == 1):
        no_lessthantest+=1
        #print(str(testindicies[0][i])+", " + str(testindicies[1][i]) +": 1.0")
    else:
        no_greaterthantest+=1
        #print(str(testindicies[0][i])+", " + str(testindicies[1][i]) +": -1.0")
print("number of Ea<Eb with a<b :" + str(no_lessthantest))
print("number of Ea>Eb with a<b:" +  str(no_greaterthantest))

1303398
number of Ea<Eb with a<b :413117
number of Ea>Eb with a<b:890281


In [6]:
X = t_cancer.values

In [7]:
CONST_l = 0.95
cancer_pairs = np.zeros((n_mirnas, n_mirnas))
diagnostic_pairs = np.zeros((n_mirnas, n_mirnas))
for i in range(0,n_mirnas):
    X_current_miRNA = X[:, i:i+1] #shape (m_samples, 1)
    X_rest_miRNA = X[:, i+1:] #shape (m_samples, n_mirnas-i-1)
    X_comparisons = X_rest_miRNA < X_current_miRNA
    X_greater = np.sum(X_comparisons, axis=0) / m_t_cancer
    X_mask_greater_correlation = np.add((X_greater >= CONST_l).nonzero(), i) #-1
    X_mask_less_correlation = np.add((X_greater <= (1-CONST_l)).nonzero(), i) #-1
    cancer_pairs[i][[X_mask_greater_correlation]] = 1
    cancer_pairs[i][[X_mask_less_correlation]] = -1
    
    #diagnostic pairs are 
    # significant[mirna1][mirna2] == 1 and cancer[mirna1][mirna2] == -1
    # or significant[mirna][mirna2] == -1 and cancer[mirna1][mirna2] == 1
    #mirna1 is fixed at i
mask1 = np.logical_and(cancer_pairs==1,significant_pairs==-1)
mask2 = np.logical_and(cancer_pairs==-1,significant_pairs==1)
diagnostic_pairs = np.logical_or(mask1, mask2)



In [8]:
testindicies2 = cancer_pairs.nonzero()
print(len(testindicies2[0]))
no_lessthantest = 0
no_greaterthantest = 0
for i in range(len(testindicies2[0])):
    if(significant_pairs[testindicies2[0][i]][testindicies2[1][i]] == 1):
        no_lessthantest+=1
        #print(str(testindicies2[0][i])+", " + str(testindicies2[1][i]) +": 1.0")
    else:
        no_greaterthantest+=1
        #print(str(testindicies2[0][i])+", " + str(testindicies[1][i]) +": -1.0")
print("number of Ea<Eb with a<b :" + str(no_lessthantest))
print("number of Ea>Eb with a<b:" +  str(no_greaterthantest))

864722
number of Ea<Eb with a<b :276218
number of Ea>Eb with a<b:588504


In [72]:
testindicies3 = diagnostic_pairs.nonzero()
print(len(testindicies3[0]))
no_lessthantest = 0
no_greaterthantest = 0
for i in range(len(testindicies3[0])):
    if(significant_pairs[testindicies3[0][i]][testindicies3[1][i]] == 1):
        no_lessthantest+=1
        #print(str(testindicies[0][i])+", " + str(testindicies[1][i]) +": 1.0")
    # in this case, diagnostic_pairs is just a binary array so honestly there is only going to be one case.
    # else: 
    #     no_greaterthantest+=1
        #print(str(testindicies[0][i])+", " + str(testindicies[1][i]) +": -1.0")
print("number of Ea<Eb with a<b :" + str(no_lessthantest))
print("number of Ea>Eb with a<b:" +  str(no_greaterthantest))

577
number of Ea<Eb with a<b :0
number of Ea>Eb with a<b:577


In [7]:
healthy_comp_arr = np.zeros((m_t_healthy, n_mirnas, n_mirnas))
healthy_arr1 = np.repeat(W[0].reshape(1,-1), n_mirnas, axis=0)
healthy_arr2 = np.repeat(W[0].reshape(-1,1), n_mirnas, axis=1)
healthy_comp_arr[0] = np.greater(healthy_arr1, healthy_arr2)

In [1]:
#make a (m_t_healthy+ m_t_cancer),n,n array with the indexes as 0/1 to indicate healthy/cancer sample
#for a sample mk (could be from healthy or cancer) you have a pair (a,b) and thus arr[a,b,mk] details 
#if Ea > Eb in that sample
healthy_comp_arr = np.zeros((m_t_healthy, n_mirnas, n_mirnas), dtype = bool)
for i in range(0,m_t_healthy):
    # you need to use np.greater(arr1, arr2)
    #where arr1 has the mirna expression the same in rows
    #and arr2 has the mirna expression the same in columns
    #more in notebook, 10/21/25 heading
    healthy_arr1 = np.repeat(W[i].reshape(1,-1), n_mirnas, axis=0)
    healthy_arr2 = np.repeat(W[i].reshape(-1,1), n_mirnas, axis=1)
    healthy_comp_arr[i] = np.greater(healthy_arr1, healthy_arr2)

NameError: name 'np' is not defined

In [ ]:
#very similar to previous
cancer_comp_arr = np.zeros((m_t_cancer, n_mirnas, n_mirnas), dtype=bool)
for i in range (0,m_t_cancer):
    cancer_arr1 = np.repeat(X[i].reshape(1,-1), n_mirnas, axis=0)
    cancer_arr2 = np.repeat(X[i].reshape(-1,1), n_mirnas, axis=1)
    cancer_comp_arr[i] = np.greater(cancer_arr1, cancer_arr2)